**Set environment**

In [1]:
suppressMessages(suppressWarnings(source("../run_config_project_sing.R")))
show_env()

You are working on        Singularity: singularity_proj_encode_fcc 
BASE DIRECTORY (FD_BASE): /data/reddylab/Kuei 
REPO DIRECTORY (FD_REPO): /data/reddylab/Kuei/repo 
WORK DIRECTORY (FD_WORK): /data/reddylab/Kuei/work 
DATA DIRECTORY (FD_DATA): /data/reddylab/Kuei/data 

You are working with      ENCODE FCC 
PATH OF PROJECT (FD_PRJ): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC 
PROJECT RESULTS (FD_RES): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC/results 
PROJECT SCRIPTS (FD_EXE): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC/scripts 
PROJECT DATA    (FD_DAT): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC/data 
PROJECT NOTE    (FD_NBK): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC/notebooks 
PROJECT DOCS    (FD_DOC): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC/docs 
PROJECT LOG     (FD_LOG): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC/log 
PROJECT REF     (FD_REF): /data/reddylab/Kuei/repo/Proj_ENCODE_FCC/references 



**Set global variable**

In [2]:
## init: region folder
vec = c("fcc_astarr_macs_input_overlap", "fcc_astarr_macs_input_union")
names(vec) = vec

VEC_TXT_FOLDER = vec
for (txt in vec) {cat(txt, "\n")}

fcc_astarr_macs_input_overlap 
fcc_astarr_macs_input_union 


In [3]:
vec = c("CRISPRi-HCRFF", "CRISPRi-Growth", "ENCODE-rE2G Benchmark")#, "CRISPR_Pool", "CRISPR_rmGrowth")
names(vec) = vec
VEC_TXT_ASSAY_OUT = vec

vec = c("ASTARR", "WSTARR", "LMPRA", "TMPRA", "ATAC")
names(vec) = vec
VEC_TXT_ASSAY_INP = vec

**Check input files**

In [4]:
txt_folder = "fcc_astarr_macs_input_union"
txt_fdiry  = file.path(FD_RES, "analysis_fcc_correlation", txt_folder)
txt_fname  = "*.rds"
txt_fglob  = file.path(txt_fdiry, txt_fname)

vec = Sys.glob(txt_fglob)
for (txt in vec) {cat(txt, "\n")}

/data/reddylab/Kuei/repo/Proj_ENCODE_FCC/results/analysis_fcc_correlation/fcc_astarr_macs_input_union/data.region.merge.group_crispr.score_starrmpra.rds 


## Import data

In [5]:
txt_foldr = "fcc_astarr_macs_input_union"
txt_fdiry = file.path(FD_RES, "analysis_fcc_correlation", txt_foldr)
txt_fname = "data.region.merge.group_crispr.score_starrmpra.rds"
txt_fpath = file.path(txt_fdiry, txt_fname)

lst = readRDS(txt_fpath)

lst_dat_region_merge = lst

res = lapply(lst, dim)
print(res)

dat = lst[[1]]
fun_display_table(head(dat, 3))

$`0.01`
[1] 627553      7

$`0.02`
[1] 627553      7

$`0.03`
[1] 627553      7

$`0.04`
[1] 627553      7

$`0.05`
[1] 627553      7

$`0.06`
[1] 627553      7

$`0.07`
[1] 627553      7

$`0.08`
[1] 627553      7

$`0.09`
[1] 627553      7

$`0.10`
[1] 627553      7

$`0.20`
[1] 627553      7

$`0.30`
[1] 627553      7

$`0.40`
[1] 627553      7

$`0.50`
[1] 627553      7



CutOff_CRISPR,Region,Assay_CRISPR,Group_CRISPR,Score_CRISPR,Assay_STARRMPRA,Score_STARRMPRA
0.01,chr10:100009096-100010466,CRISPR_Pool,0,-0.0378617,ASTARR,-1.2216566
0.01,chr10:100009096-100010466,CRISPR_Pool,0,-0.0378617,ATAC,2.3214952
0.01,chr10:100009096-100010466,CRISPR_Pool,0,-0.0378617,WSTARR,0.5896695


## Calculate Score correlations across different CRISPR threshold

In [6]:
lst = lst_dat_region_merge

### foreach CRISPR threshold
lst = lapply(lst, function(dat){
    ### foreach CRISPR assay
    lst = lapply(VEC_TXT_ASSAY_OUT, function(txt_assay_out){
        ### foreach STARR/MPRA assay
        lst = lapply(VEC_TXT_ASSAY_INP, function(txt_assay_inp){
            
            ### get the CRISPR and STARR/MPRA scores
            dat = dat %>% 
                dplyr::filter(Group_CRISPR == 1) %>%
                dplyr::filter(Assay_CRISPR    == txt_assay_out) %>%
                dplyr::filter(Assay_STARRMPRA == txt_assay_inp)
            vec_txt_score_out = dat$Score_CRISPR
            vec_txt_score_inp = dat$Score_STARRMPRA

            ### calculate spearman correlation and record number of regions
            res = data.frame(
                size = nrow(dat),
                scor = cor(vec_txt_score_out, vec_txt_score_inp, method = "spearman")
            )
            return(res)
        })
        dat = bind_rows(lst, .id = "Assay_STARRMPRA")
    })
    dat = bind_rows(lst, .id = "Assay_CRISPR")
    return(dat)
})
dat = bind_rows(lst, .id = "CutOff_CRISPR")
dat = dat %>% dplyr::mutate(CutOff_CRISPR = as.numeric(CutOff_CRISPR))

### assign and show
dat_assay_scor = dat
print(dim(dat))
fun_display_table(head(dat, 3))

[1] 210   5


CutOff_CRISPR,Assay_CRISPR,Assay_STARRMPRA,size,scor
0.01,CRISPRi-HCRFF,ASTARR,14,0.2263736
0.01,CRISPRi-HCRFF,WSTARR,14,0.2703297
0.01,CRISPRi-HCRFF,LMPRA,12,0.2276711


## Export results

In [7]:
### set file directory
txt_foldr = "fcc_astarr_macs_input_union"
txt_fdiry = file.path(FD_RES, "analysis_fcc_correlation", txt_foldr)
txt_fname = "result.cripsr_vs_starrmpra.spearman_correlation.tsv"
txt_fpath = file.path(txt_fdiry, txt_fname)

### write table
dat = dat_assay_scor
write_tsv(dat, txt_fpath)